# VN-Caption Fine-tune — Qwen2.5-1.5B-Instruct
Dataset: **captions_female_vn.jsonl** (500 ví dụ tiếng Việt — khen vẻ đẹp / vóc dáng / chuyên nghiệp / diễn xuất nữ giới, tone mạng)

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier is fine)

**Steps:**
1. Run all cells top to bottom
2. Upload `captions_female_vn.jsonl` when Cell 3 asks
3. After Cell 7 finishes, download the `.gguf` file
4. Follow the Ollama import instructions at the bottom

In [ ]:
# Cell 1 — Install (takes ~3 min)
%%capture
!pip install unsloth
!pip install -q trl datasets transformers accelerate bitsandbytes

In [ ]:
# Cell 2 — Verify GPU
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — go to Runtime > Change runtime type > T4'
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print(f'GPU: {gpu}')
print(f'VRAM: {vram} GB')
print('Ready!' if torch.cuda.is_available() else 'Switch to GPU runtime first!')

In [ ]:
# Cell 3 — Upload your dataset
from google.colab import files
print('Select captions_female_vn.jsonl from your computer...')
uploaded = files.upload()
dataset_file = list(uploaded.keys())[0]
print(f'Uploaded: {dataset_file}')

# Count examples
with open(dataset_file) as f:
    lines = [l for l in f if l.strip()]
print(f'Examples in dataset: {len(lines)}')

In [ ]:
# Cell 4 — Load base model with Unsloth
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
print('Model loaded!')

In [ ]:
# Cell 5 — Apply LoRA (parameter-efficient fine-tuning)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 6 — Load and format dataset
from datasets import load_dataset

raw = load_dataset('json', data_files=dataset_file, split='train')

def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

dataset = raw.map(format_example)

# Quick sanity check — print first example
print('=== Sample formatted example ===')
print(dataset[0]['text'][:500])
print(f'\nTotal examples: {len(dataset)}')

In [ ]:
# Cell 7 — Train (expect ~6-12 min on T4 for 500 examples)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,          # 3 passes over 500 examples
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        output_dir='./output',
        save_strategy='no',
        report_to='none',
    ),
)

print('Training started...')
trainer_stats = trainer.train()
print(f'Training done! Time: {trainer_stats.metrics["train_runtime"]:.0f}s')
print(f'Final loss: {trainer_stats.metrics["train_loss"]:.4f}')

In [ ]:
# Cell 8 — Quick quality test before export (matches training prompt format)
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def test_caption(topic):
    prompt = (
        f'Generate 1 X (Twitter) post caption about: "{topic}"\n'
        'Tone: Admiring\n\nRules:\n'
        '- Caption must be strictly under 500 characters\n'
        '- NEVER include hashtags (words starting with #) anywhere in the caption\n'
        '- Do NOT include keywords from the list in the caption text\n'
        '- Make it punchy, engaging, and share-worthy\n'
        '- Return ONLY the raw caption text — no quotes, no explanation, no JSON\n\n'
        'QUAN TRỌNG: Chủ đề bằng tiếng Việt, nên BẮT BUỘC viết caption hoàn toàn bằng tiếng Việt tự nhiên. Tuyệt đối không dùng tiếng Anh.'
    )
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=160,
        temperature=0.85,
        top_p=0.92,
        repetition_penalty=1.1,
        do_sample=True,
    )
    result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Topic: {topic}')
    print(f'Caption ({len(result)} chars): {result}')
    print()

# Test all 4 themes (Vietnamese, female praise)
test_caption('Vẻ đẹp rạng rỡ của nữ chính')
test_caption('Vóc dáng chuẩn của nữ diễn viên')
test_caption('Sự chuyên nghiệp trên phim trường')
test_caption('Diễn xuất nhập tâm của nữ chính')
test_caption('Đôi chân dài miên man')

In [ ]:
# Cell 9 — Export to GGUF (robust: merge 16-bit, build llama.cpp, quantize)
# save_pretrained_gguf often fails silently on Colab, so do it manually.
model.save_pretrained_merged('merged_model', tokenizer, save_method='merged_16bit')
print('Merged 16-bit model saved.')

!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j 4
!pip install -q gguf sentencepiece protobuf

!python llama.cpp/convert_hf_to_gguf.py merged_model --outfile vn-caption-ft-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize vn-caption-ft-f16.gguf vn-caption-ft-Q4_K_M.gguf Q4_K_M

import os
for f in os.listdir('.'):
    if f.endswith('.gguf'):
        print(f'{f}: {os.path.getsize(f)/1e6:.0f} MB')

In [ ]:
# Cell 10 — Download the GGUF (file is in root)
from google.colab import files
files.download('vn-caption-ft-Q4_K_M.gguf')

## After download — import into Ollama

1. Move the downloaded `vn-caption-ft-Q4_K_M.gguf` into your project folder, replacing the old one:
   ```
   c:\Users\phuon\VbotsCaption\
   ```

2. The `Modelfile` there already points to `./vn-caption-ft-Q4_K_M.gguf` and has the chat TEMPLATE + system prompt. No edit needed if the filename matches.

3. Rebuild the model:
   ```powershell
   cd c:\Users\phuon\VbotsCaption
   ollama create vn-caption -f Modelfile
   ```

4. Test it:
   ```powershell
   ollama run vn-caption "Generate 1 X post caption about: 'Vẻ đẹp rạng rỡ của nữ chính' Tone: Admiring. Dưới 500 ký tự, không hashtag, viết tiếng Việt, chỉ trả về caption."
   ```

**Expected loss:** good training lands between `0.8 – 1.4`. Above `2.0` means something went wrong.

The live site picks it up automatically once `ollama create` finishes — same model name, same tunnel.